# Ch 21 · Islands and Sleeping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/21-islands-and-sleep.ipynb)

Companion notebook for the chapter page `chapters/21-islands-and-sleep.html` ([page](../chapters/21-islands-and-sleep.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/21-islands-and-sleep/`). Budgets grow on a GPU (`ON_GPU`) and stay small on a CPU. Cells follow the page order, and the numbered headings are the page's section numbers. Timings are **your** numbers; CPU numbers say nothing about GPU speed.

Every scene in this notebook uses `<flag sleep="enable"/>` and `solver="Newton"` (sleep requires Newton in 4.0.1).

## 1. Islands
A *tree* is a body subtree hanging off the world body. Constraint rows (contacts, equalities, limits, friction) couple trees; an *island* is a connected group of coupled trees. `forward` runs island discovery (`island.island`, a GPU disjoint-set union over the efc rows) only when sleep is enabled; the public `mjw.island(m, d)` runs it on demand. Trees with no constraint row get `tree_island = -1`.

In [ ]:
# Example 21.1 - ex1_islands.py
"""Example 21.1 - constraint islands: who is coupled to whom.

Six boxes, one kinematic tree each. Five rest on the floor (each touches only the static floor),
box 3 hangs in the air (no constraint at all). With sleep enabled, mjw.forward runs island
discovery; with sleep disabled it does not, but the public mjw.island(m, d) still works.
C MuJoCo (mj_forward) is printed for comparison: it always computes islands.

Run: python ex1_islands.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2

Z = [0.099, 0.099, 0.099, 0.6, 0.099, 0.099]   # box 3 starts in the air


def boxes_xml(sleep):
    flag = '<flag sleep="enable"/>' if sleep else ""
    bodies = "".join(
        f'<body name="box{i}" pos="{0.5 * i:.2f} 0 {z}"><freejoint/>'
        f'<geom type="box" size=".1 .1 .1"/></body>' for i, z in enumerate(Z))
    return (f'<mujoco><option timestep="0.01" solver="Newton">{flag}</option><worldbody>'
            f'<geom name="floor" type="plane" size="5 5 .1"/>{bodies}</worldbody></mujoco>')


for sleep in (True, False):
    mjm = mujoco.MjModel.from_xml_string(boxes_xml(sleep))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    print(f"--- sleep {'enabled' if sleep else 'disabled'}: ntree {mjm.ntree}, nv {mjm.nv}")
    print("  island arrays allocated:", "tree_island", d.tree_island.shape,
          "nisland", d.nisland.shape, "efc.island", d.efc.island.shape)
    mjw.forward(m, d)
    print("  after forward:     nisland", d.nisland.numpy()[0],
          "tree_island", d.tree_island.numpy()[0], "nefc", d.nefc.numpy()[0])
    mjw.island(m, d)                        # public entry point: island discovery only
    print("  after mjw.island:  nisland", d.nisland.numpy()[0],
          "tree_island", d.tree_island.numpy()[0])

mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
print("C MuJoCo mj_forward: nisland", mjd.nisland, "tree_island", mjd.tree_island)
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 2. Contacts build islands
A stack shares one island, loose boxes get their own, and MJWarp's partition matches C MuJoCo. Then `mjw.island` timing against nworld (your GPU table on Colab).

In [ ]:
# Example 21.8 - ex8_islands_vs_contacts.py
"""Example 21.8 - contacts build islands: a stack shares one island, loose boxes get their own.

Scene: a stack of three boxes (trees 0-2), three loose boxes (trees 3-5), and two boxes touching
side by side (trees 6, 7). After forward we print tree_island and compare with C MuJoCo. Then we
time mjw.island alone (the DSU over efc rows) for a few nworld values. CPU numbers are from
Warp's CPU device; on a GPU the notebook prints its own table.

Run: python ex8_islands_vs_contacts.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLDS = [1024, 4096, 16384] if ON_GPU else [1, 4, 16]
NCALL = 100 if ON_GPU else 10

BOX = '<freejoint/><geom type="box" size=".1 .1 .1"/>'
XML = f"""
<mujoco>
  <option timestep="0.005" solver="Newton"><flag sleep="enable"/></option>
  <worldbody>
    <geom type="plane" size="5 5 .1"/>
    <body name="s0" pos="0 0 .099">{BOX}</body>
    <body name="s1" pos="0 0 .298">{BOX}</body>
    <body name="s2" pos="0 0 .497">{BOX}</body>
    <body name="a" pos=".6 0 .099">{BOX}</body>
    <body name="b" pos="1.0 0 .099">{BOX}</body>
    <body name="c" pos="1.4 0 .099">{BOX}</body>
    <body name="p" pos="0 .8 .099">{BOX}</body>
    <body name="q" pos=".199 .8 .099">{BOX}</body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2, nconmax=64, njmax=256)
mjw.forward(m, d)
print("ntree", mjm.ntree, "| nacon", d.nacon.numpy()[0], "| nefc w0", d.nefc.numpy()[0])
print("MJWarp  nisland", d.nisland.numpy()[0], "tree_island", d.tree_island.numpy()[0])
mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
print("C       nisland", mjd.nisland, "tree_island", mjd.tree_island)
print("same partition:", bool((d.tree_island.numpy()[0] == mjd.tree_island).all()))

dev = "GPU" if ON_GPU else "CPU dev box"
print(f"mjw.island timing ({dev}, {NCALL} calls each):")
for nworld in NWORLDS:
    d = mjw.make_data(mjm, nworld=nworld, nconmax=64, njmax=256)
    mjw.forward(m, d)                        # fills the efc rows the DSU walks
    mjw.island(m, d)                         # warm up (JIT)
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(NCALL):
        mjw.island(m, d)
    wp.synchronize()
    dt = (time.perf_counter() - t0) / NCALL
    print(f"  nworld {nworld:6d}: {dt * 1e3:7.3f} ms/call  {dt / nworld * 1e6:8.3f} us/world")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 4. The sleep requirements
An error gallery of every check `put_model` and `make_data` make in 4.0.1, then the same rules as a small Python function you can run on your own models.

In [ ]:
# Example 21.6 - ex6_sleep_errors.py
"""Example 21.6 - the sleep requirements, as an error gallery.

  (a) a body with sleep="init" (or "never" / "allowed")  -> NotImplementedError from put_model
  (b) sleep with solver="CG"                              -> ValueError from put_model
  (c) a flex equality constraint with sleep               -> NotImplementedError from put_model
  (d) SLEEP plus <flag island="disable"/>                 -> no error, but nothing ever sleeps
      4.0.1 pitfall: make_data then skips the compacted buffers (cM has shape (0, 0, 0)) although
      solver.solve still takes the compacted path. Pass nvmax explicitly or the step reads
      unallocated memory (a segfault on CPU). We only print the shapes, then step with nvmax=nv.
  (e) what sleep="auto" compiles to: an actuated tree gets AUTO_NEVER, a passive one AUTO_ALLOWED

Run: python ex6_sleep_errors.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2


def scene(option="", flags='sleep="enable"', body_attr="", extra=""):
    return (f'<mujoco><option timestep="0.01" {option}><flag {flags}/></option><worldbody>'
            '<geom type="plane" size="5 5 .1"/>'
            f'<body pos="0 0 .099" {body_attr}><freejoint/><geom type="box" size=".1 .1 .1"/></body>'
            f'<body pos=".5 0 .099"><freejoint/><geom type="box" size=".1 .1 .1"/></body>'
            f'{extra}</worldbody></mujoco>')


def try_put(label, xml):
    mjm = mujoco.MjModel.from_xml_string(xml)
    try:
        mjw.put_model(mjm)
        print(f"{label}: put_model OK")
    except (NotImplementedError, ValueError) as e:
        print(f"{label}: {type(e).__name__}: {e}")


for policy in ("init", "never", "allowed", "auto"):
    try_put(f'(a) body sleep="{policy}"', scene(body_attr=f'sleep="{policy}"'))
try_put('(b) solver="CG"', scene(option='solver="CG"'))
CLOTH = ('<flexcomp name="cloth" type="grid" count="3 3 1" spacing=".1 .1 .1" dim="2" '
         'pos="0 0 .6"><edge equality="true"/></flexcomp>')
try_put("(c) flex edge equality", scene(extra=CLOTH))

xml = scene(flags='sleep="enable" island="disable"')
mjm = mujoco.MjModel.from_xml_string(xml)
m = mjw.put_model(mjm)
d_bad = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
print("(d) island disabled, no nvmax:  cM shape", d_bad.cM.shape, "-> do not step this Data")
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64, nvmax=mjm.nv)
print("(d) island disabled, nvmax=nv:  cM shape", d.cM.shape)
for _ in range(100):
    mjw.step(m, d)
print("(d) after 100 resting steps: tree_asleep", d.tree_asleep.numpy()[0],
      "max |qvel|", round(float(np.abs(d.qvel.numpy()).max()), 6))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

ARM = ('<body pos="0 1 .5"><joint name="hinge" type="hinge"/><geom type="capsule" size=".03" '
       'fromto="0 0 0 .3 0 0"/></body>')
mjm = mujoco.MjModel.from_xml_string(
    scene(extra=ARM).replace("</mujoco>", '<actuator><motor joint="hinge"/></actuator></mujoco>'))
print("(e) tree_sleep_policy:", [mujoco.mjtSleepPolicy(p).name for p in mjm.tree_sleep_policy],
      "(trees: box, box, actuated arm)")

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def sleep_checklist(mjm, nvmax=None):
    """Return (ok, notes): the 4.0.1 put_model / make_data / first-step sleep rules applied to an MjModel."""
    notes = []
    sleep = bool(mjm.opt.enableflags & mujoco.mjtEnableBit.mjENBL_SLEEP)
    if not sleep:
        return True, ["SLEEP flag off: nothing to check (nvmax is ignored)"]
    sp = mujoco.mjtSleepPolicy             # 4.0.1 supports only the compiler's AUTO values
    supported = {int(sp.mjSLEEP_AUTO), int(sp.mjSLEEP_AUTO_NEVER), int(sp.mjSLEEP_AUTO_ALLOWED)}
    bad = [sp(p).name for p in np.unique(mjm.tree_sleep_policy) if int(p) not in supported]
    if bad:
        notes.append(f"put_model raises NotImplementedError: {bad} not supported")
    if (mjm.eq_type == mujoco.mjtEq.mjEQ_FLEX).any():
        notes.append("put_model raises NotImplementedError: flex equality with sleep")
    if mjm.opt.solver != mujoco.mjtSolver.mjSOL_NEWTON:
        notes.append("put_model raises ValueError: sleeping requires the Newton solver")
    discrete = mjm.opt.integrator == mujoco.mjtIntegrator.mjINT_DISCRETE
    no_island = bool(mjm.opt.disableflags & mujoco.mjtDisableBit.mjDSBL_ISLAND)
    if no_island:
        notes.append("island disabled: nothing will ever sleep"
                     + ("" if nvmax is not None else "; pass nvmax or the step reads empty buffers"))
        if discrete:
            notes.append("first forward raises ValueError: "
                         "Discrete integrator: sleep without islands not yet supported")
        if nvmax is not None and nvmax < mjm.nv:
            notes.append(f"nvmax ({nvmax}) < nv ({mjm.nv}) with islands off: NVMAX set every step, "
                         "DOFs past nvmax get qacc = 0")
    elif discrete and mjm.nflex > 0:
        # approximate: the exact check_discrete conditions use mjw Model fields (see Ch 23)
        notes.append("discrete + flex: some flex features raise at the first forward (approximate check, see Ch 23)")
    if nvmax is not None and not 0 <= nvmax <= mjm.nv:
        notes.append(f"make_data raises ValueError: nvmax ({nvmax}) must be in [0, nv ({mjm.nv})]")
    errors = [n for n in notes if "raises" in n or "empty" in n or "NVMAX" in n]
    return not errors, notes or ["all sleep requirements met"]

In [ ]:
ok_xml = ('<mujoco><option solver="Newton"><flag sleep="enable"/></option><worldbody>'
          '<geom type="plane" size="1 1 .1"/><body pos="0 0 .1"><freejoint/>'
          '<geom type="box" size=".1 .1 .1"/></body></worldbody></mujoco>')
no_island = ok_xml.replace('sleep="enable"', 'sleep="enable" island="disable"')
for label, xml, nvmax in [
        ("ok", ok_xml, None), ("CG", ok_xml.replace("Newton", "CG"), None),
        ("init", ok_xml.replace('<body pos', '<body sleep="init" pos'), None),
        ("no island", no_island, None),
        ("discrete, no island", no_island.replace('solver="Newton"', 'solver="Newton" integrator="discrete"'), 6),
        ("island off, nvmax=1", no_island, 1)]:
    mjm = mujoco.MjModel.from_xml_string(xml)
    print(label, sleep_checklist(mjm, nvmax))

## 5. Falling asleep
`tree_asleep` starts at `-(1 + mjMINAWAKE) = -11`. At the end of each step `sleep.sleep` adds 1 for every tree that is slow enough (`|dof_length * qvel| < sleep_tolerance` on every DOF, no applied force) and resets it to -11 otherwise. When every tree of an island has reached -1, `_build_cycles` links them into a circular list (values >= 0) and zeroes their qvel and qacc.

In [ ]:
# Example 21.2 - ex2_fall_asleep.py
"""Example 21.2 - boxes land, come to rest and fall asleep one island at a time.

Six tilted boxes are dropped from different heights. Every step we read tree_asleep (the per-tree
countdown / sleep-cycle array), ntree_awake and nefc. A tree's countdown starts at
-(1 + mjMINAWAKE) = -11 and rises by one per still step; at -1 its island may sleep, and the tree
then stores the next tree of its sleep cycle (>= 0).

Run: python ex2_fall_asleep.py [--dump-json PATH]
  --dump-json PATH writes world 0's box heights, pitch angles and tree_asleep values for the
  page's sleep replay (Rule D).
"""
import argparse
import json
import math

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()  # parse_known_args: also works inside Jupyter

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 160                              # enough for every box to fall asleep (default tolerance)

Z = [0.1, 0.2, 0.35, 0.5, 0.7, 0.9]      # drop heights; box i is pitched by 7*i degrees
BODIES = "".join(
    f'<body name="box{i}" pos="{0.4 * i:.2f} 0 {z}" euler="0 {7 * i} 0"><freejoint/>'
    f'<geom type="box" size=".1 .1 .1"/></body>' for i, z in enumerate(Z))
XML = ('<mujoco><option timestep="0.01" solver="Newton"><flag sleep="enable"/></option>'
       f'<worldbody><geom name="floor" type="plane" size="5 5 .1"/>{BODIES}</worldbody></mujoco>')

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
print("ntree", mjm.ntree, "| MJ_MINAWAKE", mujoco.mjMINAWAKE,
      "| sleep_tolerance", mjm.opt.sleep_tolerance)
print("initial tree_asleep", d.tree_asleep.numpy()[0])

frames, nefc_log, awake_log, first_all_asleep = [], [], [], None
for i in range(NSTEP):
    mjw.step(m, d)
    asleep = d.tree_asleep.numpy()[0]       # host read every step: demo only
    nefc = int(d.nefc.numpy()[0])
    nawake = int(d.ntree_awake.numpy()[0])
    if i % 10 == 0:
        print(f"step {i:3d}  tree_asleep {asleep}  ntree_awake {nawake}  nefc {nefc}")
    if nawake == 0 and first_all_asleep is None:
        first_all_asleep = i
    if args.dump_json and i % 2 == 0:
        q = d.qpos.numpy()[0]
        fr = []
        for t in range(mjm.ntree):
            z, qw, qy = q[7 * t + 2], q[7 * t + 3], q[7 * t + 5]
            pitch = 2.0 * math.atan2(qy, qw)
            fr.append([round(float(z), 3), round(pitch, 3), int(asleep[t])])
        frames.append(fr)
        nefc_log.append(nefc)
        awake_log.append(nawake)
wp.synchronize()

print("all trees asleep from step", first_all_asleep)
print("final tree_asleep (all worlds equal):",
      bool((d.tree_asleep.numpy() == d.tree_asleep.numpy()[0]).all()))
print("final qvel max |v|:", float(np.abs(d.qvel.numpy()).max()))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": mjm.ntree, "dt": 2 * float(mjm.opt.timestep), "kind": "tiles",
                   "labels": [f"tree {t}" for t in range(mjm.ntree)],
                   "x": [round(0.4 * t, 2) for t in range(mjm.ntree)], "half": 0.1,
                   "frames": frames, "nefc": nefc_log, "ntree_awake": awake_log},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 6. Waking up
`sleep.wake` (start of `forward`) wakes a sleeping tree that has a non-zero velocity, `qfrc_applied` or `xfrc_applied`; `sleep.wake_collision` wakes a sleeping tree that touches an awake one; `sleep.wake_equality` and `sleep.wake_tendon` do the same for active equalities and constrained tendons. Waking one tree wakes its whole sleep cycle.

In [ ]:
# Example 21.3 - ex3_wake.py
"""Example 21.3 - what wakes a sleeping tree.

Scene: a stack of two boxes (trees 0 and 1, one island) and three single boxes (trees 2, 3, 4)
spaced 0.1 m apart. We let everything fall asleep, then:
  A. apply xfrc_applied to the bottom stack box in world 0: sleep.wake sees a non-zero applied
     force and wakes the whole sleep cycle (both stack boxes), nothing else;
  B. write a velocity into box 2 in world 1: sleep.wake wakes box 2, it slides into box 3, and
     sleep.wake_collision wakes box 3 when an awake tree touches it.

Run: python ex3_wake.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2

XML = """
<mujoco>
  <option timestep="0.01" solver="Newton"><flag sleep="enable"/></option>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="stack0" pos="0 0 .1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="stack1" pos="0 0 .3"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="box2" pos=".6 0 .1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="box3" pos=".9 0 .1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="box4" pos="1.2 0 .1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)


def awake_trees(world):
    return [t for t, v in enumerate(d.tree_asleep.numpy()[world]) if v < 0]


for i in range(300):                              # settle until every tree sleeps
    mjw.step(m, d)
    if d.ntree_awake.numpy().max() == 0:
        break
print(f"all asleep after {i + 1} steps; tree_asleep w0 {d.tree_asleep.numpy()[0]}")
print("  (stack0 -> stack1 -> stack0 is one sleep cycle; 2, 3, 4 are self-cycles)")

# A. a force on the bottom stack box, world 0 only
xfrc = np.zeros((NWORLD, mjm.nbody, 6), dtype=np.float32)
xfrc[0, mjm.body("stack0").id, 0] = 5.0           # 5 N along +x
d.xfrc_applied.assign(xfrc)
mjw.step(m, d)
print("A. xfrc_applied on stack0 (w0): awake w0", awake_trees(0), "| awake w1", awake_trees(1))
d.xfrc_applied.zero_()

# B. a velocity written into box 2, world 1 only
qvel = d.qvel.numpy()
qvel[1, mjm.jnt_dofadr[mjm.body("box2").jntadr[0]]] = 2.0   # vx = 2 m/s
d.qvel.assign(qvel)
seen = []
for k in range(12):
    mjw.step(m, d)
    now = awake_trees(1)
    if now != (seen[-1][1] if seen else None):
        seen.append((k, now))
for k, now in seen:
    print(f"B. step {k:2d} after the qvel write: awake w1 {now}")

for i in range(300):                              # everything settles again
    mjw.step(m, d)
    if d.ntree_awake.numpy().max() == 0:
        break
print(f"asleep again after {i + 1} more steps; tree_asleep w1 {d.tree_asleep.numpy()[1]}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 7. Starting asleep without sleep="init"
The `--init_asleep` pattern from `_src/cli.py`: write `mjd.tree_asleep[:] = np.arange(ntree)` on the host, then call `put_data`.

In [ ]:
# Example 21.7 - ex7_init_asleep.py
"""Example 21.7 - start every world asleep without sleep="init" (the cli.py --init_asleep pattern).

4.0.1 rejects body sleep="init", so mujoco_warp/_src/cli.py does this instead: write
mjd.tree_asleep[:] = np.arange(mjm.ntree) on the host (each tree is a sleep cycle of itself) and
then call put_data, which copies the host tree_asleep before it runs mj_kinematics (that call
would reset it). Every world starts asleep; a push wakes only the pushed tree.

Run: python ex7_init_asleep.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2

BODIES = "".join(
    f'<body name="box{i}" pos="{0.5 * i:.2f} 0 0.099"><freejoint/>'
    f'<geom type="box" size=".1 .1 .1"/></body>' for i in range(6))
XML = ('<mujoco><option timestep="0.01" solver="Newton"><flag sleep="enable"/></option>'
       f'<worldbody><geom type="plane" size="5 5 .1"/>{BODIES}</worldbody></mujoco>')

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)

mjd = mujoco.MjData(mjm)
mjd.tree_asleep[:] = np.arange(mjm.ntree, dtype=np.int32)    # the --init_asleep line
d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=32, njmax=128)
print("after put_data: tree_asleep", d.tree_asleep.numpy()[0],
      "tree_awake", d.tree_awake.numpy()[0])

mjw.step(m, d)
print("after 1 step:   tree_asleep", d.tree_asleep.numpy()[0],
      "ntree_awake", d.ntree_awake.numpy(), "nefc", d.nefc.numpy(), "nacon", d.nacon.numpy())

xfrc = np.zeros((NWORLD, mjm.nbody, 6), dtype=np.float32)
xfrc[0, mjm.body("box4").id, 2] = 200.0                      # lift box 4 in world 0
d.xfrc_applied.assign(xfrc)
mjw.step(m, d)
print("after a push:   tree_asleep", d.tree_asleep.numpy()[0],
      "ntree_awake", d.ntree_awake.numpy(), "nefc", d.nefc.numpy())

ref = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
print("for comparison, make_data starts awake:", ref.tree_asleep.numpy()[0])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 8. nvmax: the capacity of the compacted solve
With sleep on, the awake DOFs are packed into dense buffers of `nvmax_pad` columns (`island.update_active_dofs`). More awake DOFs than `nvmax` sets the NVMAX overflow bit, and the DOFs that did not fit get zero acceleration. With sleep off, nvmax is ignored.

In [ ]:
# Example 21.4 - ex4_nvmax.py
"""Example 21.4 - nvmax: capacity of the compacted (awake-DOF) solve.

With sleep enabled, every step packs the DOFs of awake trees into a dense buffer of nvmax_pad
columns (island.update_active_dofs) and runs the Newton solve there. If more DOFs are awake than
nvmax, the NVMAX overflow bit is set. Part 1: the 6-box scene (nv = 36) with nvmax = 12 while all
boxes are awake. Part 2: the right size. Part 3: the same small nvmax with sleep disabled: the
compacted path never runs, so no bit is ever set (nvmax only matters with SLEEP).
In part 1 the DOFs that did not fit are frozen (qacc 0), so those boxes look still and fall asleep:
the physics is wrong and nothing but the overflow bit tells you.

Run: python ex4_nvmax.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 20


def boxes_xml(sleep):
    flag = '<flag sleep="enable"/>' if sleep else ""
    bodies = "".join(
        f'<body name="box{i}" pos="{0.5 * i:.2f} 0 0.099"><freejoint/>'
        f'<geom type="box" size=".1 .1 .1"/></body>' for i in range(6))
    return (f'<mujoco><option timestep="0.01" solver="Newton">{flag}</option><worldbody>'
            f'<geom name="floor" type="plane" size="5 5 .1"/>{bodies}</worldbody></mujoco>')


def decode(bits):
    return [o.name for o in mjw.OverflowType if bits & o] or ["none"]


def run(sleep, nvmax, label):
    mjm = mujoco.MjModel.from_xml_string(boxes_xml(sleep))
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0                 # we decode the bits ourselves (no printf per world)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128, nvmax=nvmax)
    for _ in range(NSTEP):
        mjw.step(m, d)
    bits = int(np.bitwise_or.reduce(d.overflow.numpy()))
    print(f"{label}: nv {mjm.nv}, nvmax {d.nvmax}, nvmax_pad {d.nvmax_pad}, "
          f"ncdof {d.ncdof.numpy()[0]}, ntree_awake {d.ntree_awake.numpy()[0]} "
          f"-> overflow {decode(bits)}")
    print("   tree_asleep w0", d.tree_asleep.numpy()[0])


run(True, 12, "1. sleep on,  nvmax too small")
run(True, 36, "2. sleep on,  nvmax = nv     ")
run(False, 12, "3. sleep off, nvmax too small")

try:
    mjw.make_data(mujoco.MjModel.from_xml_string(boxes_xml(True)), nvmax=40)
except ValueError as e:
    print("nvmax > nv:", e)

## 10. What sleep costs and buys
On a GPU this cell prints your own numbers. On a CPU the compacted Newton loop runs all `opt.iterations` (no graph conditional off CUDA), so sleep looks slow there; what carries over is the work count: asleep worlds have `nacon = 0` and `nefc = 0`.

In [ ]:
# Example 21.5 - ex5_sleep_cost.py
"""Example 21.5 - what sleep buys: step time of a resting clutter scene, awake vs asleep.

A 4 x 4 grid of resting boxes (16 trees, nv = 96). We time mjw.step in three states:
  off    sleep disabled (the normal pipeline),
  awake  sleep enabled, first steps (every countdown starts at -11, so all trees are awake),
  asleep sleep enabled, after every tree has fallen asleep.
CPU numbers are from Warp's CPU device and say nothing about GPU speed; the notebook prints your
own GPU numbers. On CPU the sleep path is slower: solver.solve_compact turns graph_conditional off
off-CUDA, so the compacted Newton loop runs all opt.iterations instead of stopping at convergence.
What carries over to the GPU is the work count: asleep worlds have no contacts and no efc rows.

Run: python ex5_sleep_cost.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NTIME = 100 if ON_GPU else 8           # timed steps per state


def clutter_xml(sleep):
    flag = '<flag sleep="enable"/>' if sleep else ""
    bodies = "".join(
        f'<body pos="{0.3 * (i % 4):.2f} {0.3 * (i // 4):.2f} 0.099"><freejoint/>'
        f'<geom type="box" size=".1 .1 .1"/></body>' for i in range(16))
    return (f'<mujoco><option timestep="0.01" solver="Newton">{flag}</option><worldbody>'
            f'<geom type="plane" size="5 5 .1"/>{bodies}</worldbody></mujoco>')


def time_steps(m, d, n):
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        mjw.step(m, d)
    wp.synchronize()
    return (time.perf_counter() - t0) / n


results, work = {}, {}
for sleep in (False, True):
    mjm = mujoco.MjModel.from_xml_string(clutter_xml(sleep))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=96, njmax=512)
    warm = mjw.make_data(mjm, nworld=NWORLD, nconmax=96, njmax=512)
    for _ in range(3):
        mjw.step(m, warm)                  # JIT compile + warm up on a throwaway Data
    if not sleep:
        results["off"] = (time_steps(m, d, NTIME), d)
        work["off"] = (d.nacon.numpy()[0], d.nefc.numpy()[0])
        continue
    results["awake"] = (time_steps(m, d, NTIME), d)   # countdowns still running: all awake
    work["awake"] = (d.nacon.numpy()[0], d.nefc.numpy()[0])
    print("awake state:  ntree_awake", d.ntree_awake.numpy()[0])
    for _ in range(200):
        mjw.step(m, d)
        if d.ntree_awake.numpy().max() == 0:
            break
    mjw.step(m, d)                          # one step in the asleep state before timing
    print("asleep state: ntree_awake", d.ntree_awake.numpy()[0])
    results["asleep"] = (time_steps(m, d, NTIME), d)
    work["asleep"] = (d.nacon.numpy()[0], d.nefc.numpy()[0])

dev = "GPU" if ON_GPU else "CPU dev box"
for k, (dt, _) in results.items():
    nacon, nefc = work[k]
    print(f"{k:7s} {dt * 1e3:8.2f} ms/step  {NWORLD / dt:9.0f} world-steps/s  ({dev})"
          f"  nacon (all worlds) {nacon:4d}  nefc w0 {nefc:4d}")
print(f"asleep vs off: {results['off'][0] / results['asleep'][0]:.2f}x speed-up ({dev})")
print(f"asleep vs awake (sleep on): {results['awake'][0] / results['asleep'][0]:.2f}x speed-up")
print("overflow:", "none" if not any(r[1].overflow.numpy().any() for r in results.values())
      else "set")

## Exercise 21.1: size nvmax
All six boxes start asleep. Boxes 0 and 1 are pushed in every world and nothing else ever moves. Pick the smallest `NVMAX` that never sets the NVMAX bit, and check that 6 is too small.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BODIES = "".join(
    f'<body name="box{i}" pos="{0.5 * i:.2f} 0 0.099"><freejoint/>'
    f'<geom type="box" size=".1 .1 .1"/></body>' for i in range(6))
XML = ('<mujoco><option timestep="0.01" solver="Newton"><flag sleep="enable"/></option>'
       f'<worldbody><geom type="plane" size="5 5 .1"/>{BODIES}</worldbody></mujoco>')
mjm = mujoco.MjModel.from_xml_string(XML)
NWORLD = 4


def run(nvmax, nstep=60):
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    mjd = mujoco.MjData(mjm)
    mjd.tree_asleep[:] = np.arange(mjm.ntree, dtype=np.int32)      # start asleep
    d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=32, njmax=128, nvmax=nvmax)
    xfrc = np.zeros((NWORLD, mjm.nbody, 6), dtype=np.float32)
    xfrc[:, mjm.body("box0").id, 2] = 150.0                          # lift box 0 ...
    xfrc[:, mjm.body("box1").id, 0] = 30.0                           # ... and slide box 1
    d.xfrc_applied.assign(xfrc)
    peak = 0
    for i in range(nstep):
        if i == 5:
            d.xfrc_applied.zero_()
        mjw.step(m, d)
        peak = max(peak, int(d.nv_awake.numpy().max()))
    bits = int(np.bitwise_or.reduce(d.overflow.numpy()))
    return bits, peak


NVMAX = ...          # your choice: the smallest capacity that never overflows
bits, peak = run(NVMAX)
print("nvmax", NVMAX, "peak awake dofs", peak, "overflow bits", bits)
assert NVMAX < mjm.nv
assert not bits & mjw.OverflowType.NVMAX
bits6, _ = run(6)
print("nvmax 6 -> overflow", [o.name for o in mjw.OverflowType if bits6 & o])
assert bits6 & mjw.OverflowType.NVMAX
print("OK")

In [ ]:
#@title Exercise 21.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BODIES = "".join(
    f'<body name="box{i}" pos="{0.5 * i:.2f} 0 0.099"><freejoint/>'
    f'<geom type="box" size=".1 .1 .1"/></body>' for i in range(6))
XML = ('<mujoco><option timestep="0.01" solver="Newton"><flag sleep="enable"/></option>'
       f'<worldbody><geom type="plane" size="5 5 .1"/>{BODIES}</worldbody></mujoco>')
mjm = mujoco.MjModel.from_xml_string(XML)
NWORLD = 4


def run(nvmax, nstep=60):
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    mjd = mujoco.MjData(mjm)
    mjd.tree_asleep[:] = np.arange(mjm.ntree, dtype=np.int32)      # start asleep
    d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=32, njmax=128, nvmax=nvmax)
    xfrc = np.zeros((NWORLD, mjm.nbody, 6), dtype=np.float32)
    xfrc[:, mjm.body("box0").id, 2] = 150.0                          # lift box 0 ...
    xfrc[:, mjm.body("box1").id, 0] = 30.0                           # ... and slide box 1
    d.xfrc_applied.assign(xfrc)
    peak = 0
    for i in range(nstep):
        if i == 5:
            d.xfrc_applied.zero_()
        mjw.step(m, d)
        peak = max(peak, int(d.nv_awake.numpy().max()))
    bits = int(np.bitwise_or.reduce(d.overflow.numpy()))
    return bits, peak


NVMAX = 12
bits, peak = run(NVMAX)
print("nvmax", NVMAX, "peak awake dofs", peak, "overflow bits", bits)
assert NVMAX < mjm.nv
assert not bits & mjw.OverflowType.NVMAX
bits6, _ = run(6)
print("nvmax 6 -> overflow", [o.name for o in mjw.OverflowType if bits6 & o])
assert bits6 & mjw.OverflowType.NVMAX
print("OK")

## Exercise 21.2: all_asleep(d)
Write `all_asleep(d)` and assert that a settling three-box scene reaches it within 200 steps.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

Z = [0.1, 0.3, 0.5]
BODIES = "".join(
    f'<body pos="{0.4 * i:.2f} 0 {z}"><freejoint/><geom type="box" size=".1 .1 .1"/></body>'
    for i, z in enumerate(Z))
XML = ('<mujoco><option timestep="0.01" solver="Newton"><flag sleep="enable"/></option>'
       f'<worldbody><geom type="plane" size="5 5 .1"/>{BODIES}</worldbody></mujoco>')
mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=16, njmax=64)


def all_asleep(d):
    """True when every tree in every world is asleep."""
    ...      # read d.tree_asleep (or d.ntree_awake) on the host


assert not all_asleep(d)                 # make_data starts every tree awake
N = 200
for i in range(N):
    mjw.step(m, d)
    if all_asleep(d):
        break
print("all asleep after", i + 1, "steps")
assert all_asleep(d)
assert (d.ntree_awake.numpy() == 0).all()
print("OK")

In [ ]:
#@title Exercise 21.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

Z = [0.1, 0.3, 0.5]
BODIES = "".join(
    f'<body pos="{0.4 * i:.2f} 0 {z}"><freejoint/><geom type="box" size=".1 .1 .1"/></body>'
    for i, z in enumerate(Z))
XML = ('<mujoco><option timestep="0.01" solver="Newton"><flag sleep="enable"/></option>'
       f'<worldbody><geom type="plane" size="5 5 .1"/>{BODIES}</worldbody></mujoco>')
mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=16, njmax=64)


def all_asleep(d):
    """True when every tree in every world is asleep."""
    return bool((d.tree_asleep.numpy() >= 0).all())


assert not all_asleep(d)                 # make_data starts every tree awake
N = 200
for i in range(N):
    mjw.step(m, d)
    if all_asleep(d):
        break
print("all asleep after", i + 1, "steps")
assert all_asleep(d)
assert (d.ntree_awake.numpy() == 0).all()
print("OK")

## Stretch: throughput against the fraction of moving objects
On a GPU, build a 64-box clutter scene, start every world asleep, and keep a fraction f of the boxes moving with `xfrc_applied`. Measure world-steps/s for f in 0, 0.1, 0.25, 0.5 and 1 with sleep on (nvmax sized for f) and off, and plot the ratio.